# Track 2 · 04 REVE with BCI-domain pretraining: MOABB LoRA pretraining, then PROTEUS LoRA fine-tuning

**Idea.** REVE-base saw mostly clinical EEG (about 82% of its hours; BCI about 1%). Stage 1 adapts it, with LoRA and no labels, to motor-imagery EEG from MOABB, using REVE's own masked-reconstruction task. Stage 2 is the recipe of `track2_03` (LoRA + per-electrode head, causal session scaling, scored on the sealed-like folds) on top of the stage-1 encoder.

| step | what | time |
|---|---|---|
| Stage 1 | frozen REVE + LoRA + new decoder, masked reconstruction on MOABB; merged encoder and card **saved to Drive** | MOABB download + about 10-20 min |
| Stage 2 | sealed-like folds with the stage-1 encoder (`USE_STAGE1`), curves | about 1 h |
| Zip | one final run on all 10 people, harness-style checks, zip | about 10 min |

**Reusing a finished stage 1:** set `RUN_STAGE1 = False`; stage 2 loads `STAGE1_NAME`'s saved encoder from Drive. **Baseline:** `USE_STAGE1 = False` reproduces `track2_03` Step 2 (unseen 0.502, seen 0.589, mix 0.545).

External data: MOABB datasets are public, but licences differ per dataset. Check each before relying on it; the card file written in stage 1 lists what was used for the method-description declaration.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

# What runs
RUN_STAGE1 = True       # Stage 1: MOABB LoRA pretraining. False = reuse the saved encoder named below.
USE_STAGE1 = True       # Stage 2 starts from the stage-1 encoder. False = plain REVE-base (the track2_03 baseline).
STAGE1_NAME = 'moabb_v1'
RUN_REVE_FOLDS = True   # Stage 2: sealed-like folds (about 1 h)
ZIP_REVE = False        # Stage 2: one final run on all 10 people, write and check the zip
SEED = 33
LATE_RUNS = 2           # sealed-like folds: a single-session person is tested on their last LATE_RUNS runs

# ---- Stage 1 (self-supervised, MOABB)
STAGE1_DATASETS = ['BNCI2014_001', 'PhysionetMI', 'Cho2017', 'Schirrmeister2017']   # check each dataset's licence
STAGE1_MAX_SUBJECTS = 20      # per dataset (downloads are large; PhysionetMI has 109 people)
STAGE1_MAX_WINDOWS = 6000     # per dataset, random subsample of the 2 s windows
STAGE1_STEPS = 6000
STAGE1_BATCH = 64
STAGE1_EVAL_EVERY = 250
STAGE1_VAL_WINDOWS = 512      # held-out-people windows per dataset used for the reconstruction check
S1_MASK_RATIO = 0.5           # share of tokens (electrode x 1 s patch) hidden
S1_RANK, S1_RANK_FFN, S1_ALPHA_FFN = 8, 8, 8     # stage-1 adapters (stage 2 keeps its own, smaller ones below)
S1_LORA_DROPOUT = 0.0
S1_LR_ADAPTER, S1_LR_DEC = 1e-4, 1e-3
S1_WD = 0.01
S1_WARMUP = 500               # steps with the decoder training and the adapters frozen
S1_DEC_DIM, S1_DEC_DEPTH = 256, 4

# ---- Stage 2: the track2_03 REVE recipe
EPOCHS = 40
KEEP = 'last'
LR_SCHEDULE = 'cosine'
BATCH = 64
LABEL_SMOOTHING = 0.12
WEIGHT_DECAY = 0.0
CROP_S = 2.0
HEAD_ONLY_EPOCHS = 1
LR_ADAPTER, LR_HEAD = 2e-4, 1e-3
LORA_RANK = 4
LORA_DROPOUT = 0.10
LORA_RANK_FFN = 8
LORA_ALPHA_FFN = 4
HEAD_SHRINK = 16
HEAD_DIMS = (128, 32)
HEAD_DROPOUT_MID = 0.35
HEAD_NORM = True
HEAD_INIT = 'kaiming'
HEAD_DROPOUT = 0.1
SESSION_N0 = 5.0
SEALED_CV_SEEDS = 1
FINAL_SEEDS = 1

AUG_MASK_RATIO = 0.15
AUG_MASK_NOISE = 1.0
AUG_MASK_RADIUS = 0.05
AUG_GAIN_SD = 0.1
AUG_SHIFT_S = 0.1

PLOT_LOSS_YLIM = (0.0, 2.4)
PLOT_ACC_YLIM = (0.30, 0.70)

print(f'NOTEBOOK VERSION: track2_04 REVE + MOABB stage 1   stage1={STAGE1_NAME} (run={RUN_STAGE1}, use={USE_STAGE1})   epochs {EPOCHS}')

In [ ]:
%pip install -q 'braindecode==1.8.1' huggingface_hub safetensors moabb einops

import importlib
for mod in ('braindecode', 'torch', 'numpy', 'sklearn'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')

In [ ]:
import time
from pathlib import Path
import numpy as np, torch
from braindecode.models import REVE
import torch.nn.functional as F


def standardize(x):
    # Windows arrive already scaled by the causal session norm, so this only converts the type.
    return torch.as_tensor(x).float()


def resample_fft(x, n_out):
    """Resample along time via the rfft spectrum (same function the solvers use)."""
    n_in = x.shape[-1]
    if n_in == n_out:
        return x
    spec = torch.fft.rfft(x.float(), dim=-1)
    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)
    keep = min(spec.shape[-1], out.shape[-1])
    out[..., :keep] = spec[..., :keep]
    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)


def load_reve(n_classes):
    """Pretrained brain-bzh/reve-base, built the way Track 1 builds it (200 Hz, 1 s
    patches). The task head is dropped; ours is added by the caller."""
    net = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=n_classes, n_chans=N_CHANS,
                               n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    net.final_layer = torch.nn.Identity()
    if USE_STAGE1:                       # start from the MOABB-adapted encoder saved by stage 1
        p1 = Path(ROOT) / 'Track2' / 'stage1' / f'{STAGE1_NAME}_encoder_merged.pt'
        assert p1.exists(), f'{p1} not found: run stage 1 first (RUN_STAGE1 = True) or set USE_STAGE1 = False'
        res = net.load_state_dict(torch.load(p1, map_location='cpu', weights_only=True), strict=False)
        assert not res.unexpected_keys and not res.missing_keys, (res.unexpected_keys[:3], res.missing_keys[:3])
        print(f'loaded stage-1 encoder {p1.name}')
    bank = {k.casefold() for k in net._position_bank.mapping}
    missing = [c for c in CH if c.casefold() not in bank]
    print(f'REVE pretrained loaded; electrodes without a known position: {missing or "none"}')
    assert not missing, 'REVE would mis-size its input; fix the channel names first'
    return net


class LoRALinear(torch.nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.drop = torch.nn.Dropout(dropout)    # adapter branch only; the frozen base is untouched
        for p_ in self.base.parameters():
            p_.requires_grad_(False)
        self.A = torch.nn.Parameter(torch.zeros(r, base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
        torch.nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

    @property
    def weight(self):
        """Effective weight, W + scale * B A. braindecode >= 2.0 reads `net[1].weight` of the
        feed-forward GEGLU directly instead of calling the module, which would skip the
        adapter; with this property the adapter is applied there too. (The input dropout of
        this adapter does not apply on that path.)"""
        return self.base.weight + (self.B @ self.A) * self.scale


class PerElectrodeHead(torch.nn.Module):
    """Shrink each electrode's embedding with weights shared across electrodes (one linear
    layer, or a funnel of several), then a final layer with separate weights per electrode.
    norm: a LayerNorm after every funnel linear layer, which keeps the signal size steady.
    init: 'kaiming' gives the funnel layers He starting weights (PyTorch's default shrinks the
    signal at every layer)."""

    def __init__(self, n_chans, embed_dim, dims, n_outputs, dropout, mid_dropout=0.0,
                 norm=False, init='default'):
        super().__init__()
        dims = [dims] if isinstance(dims, int) else list(dims)
        self.norm = torch.nn.LayerNorm(embed_dim)
        if len(dims) == 1 and not norm:
            self.shrink = torch.nn.Linear(embed_dim, dims[0])        # keeps the older key names
        else:
            layers, prev = [], embed_dim
            for i, d in enumerate(dims):
                layers.append(torch.nn.Linear(prev, d))
                prev = d
                if norm:
                    layers.append(torch.nn.LayerNorm(d))
                if i < len(dims) - 1:
                    layers += [torch.nn.GELU(), torch.nn.Dropout(mid_dropout)]
            self.shrink = torch.nn.Sequential(*layers)
        if init == 'kaiming':
            for m in self.shrink.modules():
                if isinstance(m, torch.nn.Linear):
                    torch.nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                    torch.nn.init.zeros_(m.bias)
        self.drop = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(n_chans * dims[-1], n_outputs)

    def forward(self, tokens):                    # (B, chan, embed)
        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))
        return self.out(z.flatten(1))


class ReveLoRA(torch.nn.Module):
    def __init__(self, n_classes, device, rank=LORA_RANK):
        super().__init__()
        self.net = load_reve(n_classes)
        self.net.final_layer = torch.nn.Identity()
        self.head = PerElectrodeHead(N_CHANS, self.net.embed_dim, HEAD_DIMS if HEAD_DIMS else HEAD_SHRINK,
                                     n_classes, HEAD_DROPOUT, HEAD_DROPOUT_MID, HEAD_NORM, HEAD_INIT)
        self.n_lora = 0
        for name, m in list(self.net.named_modules()):
            if isinstance(m, torch.nn.Linear) and name.split('.')[-1] in ('to_qkv', 'to_out'):
                parent = self.net.get_submodule(name.rsplit('.', 1)[0])
                setattr(parent, name.split('.')[-1], LoRALinear(m, rank, 2 * rank, LORA_DROPOUT))
                self.n_lora += 1
        self.n_ffn = 0
        if LORA_RANK_FFN:        # the two feed-forward matrices of each block, at net.1 and net.3
            for name, m in list(self.net.named_modules()):
                if (isinstance(m, torch.nn.Linear) and name.startswith('transformer.layers.')
                        and name.endswith(('.net.1', '.net.3'))):
                    parent = self.net.get_submodule(name.rsplit('.', 1)[0])
                    setattr(parent, name.split('.')[-1],
                            LoRALinear(m, LORA_RANK_FFN, LORA_ALPHA_FFN, LORA_DROPOUT))
                    self.n_ffn += 1
        for p_ in self.net.parameters():
            p_.requires_grad_(False)
        for n_, p_ in self.net.named_parameters():
            if n_.endswith(('.A', '.B')):
                p_.requires_grad_(True)
        self.to(device).eval()

    def forward(self, x):
        x = resample_fft(x.float(), int(round(x.shape[-1] * 200.0 / SFREQ)))   # REVE is 200 Hz
        feats = self.net(x, return_features=True)['features']      # (B, chan, patch, embed)
        return self.head(feats.mean(dim=2))                        # average the time patches only


def train_reve_lora(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = ReveLoRA(N_CLASSES, DEVICE)
    adapters = [p_ for n_, p_ in net.net.named_parameters() if p_.requires_grad]
    opt = torch.optim.Adam([dict(params=adapters, lr=LR_ADAPTER),
                            dict(params=list(net.head.parameters()), lr=LR_HEAD)],
                           weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        n_ad = sum(p_.numel() for n_, p_ in net.net.named_parameters() if p_.requires_grad)
        print(f'{label}: REVE LoRA {n_ad:,} adapter params ({net.n_lora} attention + {net.n_ffn} feed-forward), '
              'per-electrode head, '
              f'{len(ytr):,} windows, lr {LR_SCHEDULE}, seed {seed}, head-only epochs {HEAD_ONLY_EPOCHS}, '
              f'lr adapters {LR_ADAPTER:g} / head {LR_HEAD:g}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        for p_ in adapters:                      # staged: head alone first, then adapters too
            p_.requires_grad_(ep > HEAD_ONLY_EPOCHS)
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(aug(standardize(Xtr_t[b].to(DEVICE)))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()
                          if k.endswith(('.A', '.B')) or k.startswith('head.')}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.load_state_dict(best_state, strict=False)
    return net, hist, probs


print('ready')

## Stage 1 — BCI-domain LoRA pretraining on MOABB (no labels)

REVE-base was pretrained on mostly clinical EEG (about 82% of its hours; BCI about 1%). Here REVE stays **frozen** and only LoRA adapters (attention + feed-forward) and a new light decoder learn to reconstruct hidden parts of motor-imagery EEG from MOABB. The task is REVE's own: hide tokens (electrode x 1 s patch) in blocks, encode only the visible ones, reconstruct the hidden raw patches (L1 loss). The official decoder is not in the released encoder, so ours is trained from scratch, with a decoder-only warm-up so its random start does not push noise into the adapters.

When it ends, the best adapters are merged into the encoder (W + B·A, exact) and **saved to Drive** with a card listing the datasets (the challenge asks for external corpora to be declared). Stage 2 can then load the saved encoder directly: set `RUN_STAGE1 = False` and `USE_STAGE1 = True`. The original REVE file is untouched, so stage 1 can always be undone (`USE_STAGE1 = False`).

In [ ]:
# Stage 1 data: MOABB motor-imagery epochs, cut into 2 s windows at 200 Hz (REVE's rate; the same length stage 2 feeds it),
# scaled by the causal per-session scheme used in stage 2 (units cancel), one block per dataset because channel sets differ.
import time
from pathlib import Path
if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
import numpy as np, torch

STAGE1_DIR = Path(ROOT) / 'Track2' / 'stage1'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
S1_WIN = 400                                   # 2 s at 200 Hz


def session_norm(Xg, Sg, prior_var, n0=5.0):
    """Causal per-session scaling (same as stage 2): remove the window mean, divide per electrode by the root of the
    mean power of EARLIER windows of the session blended with a prior of n0 windows. Labels are never used."""
    out = np.empty(Xg.shape, dtype='float32')
    _, first = np.unique(Sg, return_index=True)
    for s_ in Sg[np.sort(first)]:
        idx = np.flatnonzero(Sg == s_)
        x = Xg[idx].astype('float64')
        x = x - x.mean(-1, keepdims=True)
        v = x.var(-1)
        before = np.cumsum(v, 0) - v
        t = np.arange(len(idx))[:, None]
        m = (n0 * prior_var[None] + before) / (n0 + t)
        out[idx] = (x / np.sqrt(m)[..., None]).astype('float32')
    return out


def load_moabb_block(name, bank, max_subjects, max_windows, seed=0):
    """-> dict(X (N, C, 400) session-scaled float32, chs, subj (N,), doi). Channels REVE has no position for are dropped."""
    import moabb.datasets as mdatasets
    from moabb.paradigms import MotorImagery
    ds = getattr(mdatasets, name)()
    para = MotorImagery(resample=200.0, fmin=1.0, fmax=45.0)       # all motor-imagery classes of the dataset
    xs, subj, ses = [], [], []
    chs = None
    for s_ in ds.subject_list[:max_subjects]:
        ep, y, md = para.get_data(ds, subjects=[s_], return_epochs=True)
        names = list(ep.ch_names)
        keep = [i for i, c in enumerate(names) if c.casefold() in bank]
        if chs is None:
            chs = [names[i] for i in keep]
        assert [names[i] for i in keep] == chs, f'{name}: channel set changed between subjects'
        data = ep.get_data()[:, keep] * 1e6                         # volts -> microvolts
        for k in range(data.shape[-1] // S1_WIN):                   # non-overlapping 2 s windows of each epoch
            xs.append(data[..., k * S1_WIN:(k + 1) * S1_WIN]); subj.append(np.full(len(data), s_))
            ses.append(np.array([f'{name}_{s_}_{a}_{b}' for a, b in zip(md['session'], md['run'])]))
    X = np.concatenate(xs).astype('float32')
    subj, ses = np.concatenate(subj), np.concatenate(ses)
    order = np.lexsort((np.arange(len(X)), ses))                    # keep time order inside each session
    X, subj, ses = X[order], subj[order], ses[order]
    if len(X) > max_windows:                                        # random subsample, order inside sessions kept
        sel = np.sort(np.random.default_rng(seed).choice(len(X), max_windows, replace=False))
        X, subj, ses = X[sel], subj[sel], ses[sel]
    prior = X.astype('float64').var(-1).mean(0)
    return dict(X=session_norm(X, ses, prior, SESSION_N0), chs=chs, subj=subj, doi=str(getattr(ds, 'doi', '')))


if RUN_STAGE1:
    _tmp = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=3, n_chans=3, n_times=200, sfreq=200.0,
                                chs_info=[{'ch_name': c} for c in ('C3', 'Cz', 'C4')])
    BANK = {k.casefold(): k for k in _tmp._position_bank.mapping}
    S1 = {}
    for nm_ in STAGE1_DATASETS:
        t0 = time.time()
        S1[nm_] = load_moabb_block(nm_, BANK, STAGE1_MAX_SUBJECTS, STAGE1_MAX_WINDOWS)
        b_ = S1[nm_]
        b_['pos'] = _tmp.get_positions([BANK[c.casefold()] for c in b_['chs']]).detach().cpu().numpy().astype(float)
        people = sorted(set(b_['subj'])); n_val = max(1, len(people) // 10)
        val_p = people[-n_val:]                                     # last 10% of subjects: held-out reconstruction check
        b_['val'] = np.isin(b_['subj'], val_p)
        print(f'{nm_:20s} {len(b_["X"]):6,d} windows  {len(people):3d} people ({n_val} held out)  {len(b_["chs"]):3d} channels  {time.time() - t0:.0f}s')
    del _tmp
else:
    print('Stage 1 data skipped (RUN_STAGE1 = False)')

In [ ]:
# Stage 1 model: frozen REVE + LoRA adapters, a new light decoder, REVE's masked-reconstruction objective.
import numpy as np


class ElectrodeMasker:
    """positions: (C, 3) electrode coordinates in metres (REVE's position bank is in metres,
    head radius about 0.095). n_time: number of time tokens per window."""

    def __init__(self, positions, n_time=1, ratio=0.2, ratio_concentration=8.0,
                 spatial_scale=0.05, temporal_scale=1.0, scale_sigma=0.5,
                 n_bumps=(2, 5), noise=0.3, min_visible_frac=0.4, balance=True):
        pos = np.asarray(positions, dtype=float)
        self.C, self.T = len(pos), int(n_time)
        self.D = np.linalg.norm(pos[:, None] - pos[None], axis=-1)       # (C, C) metres
        self.ratio, self.conc = float(ratio), float(ratio_concentration)
        self.sp, self.tp, self.sig = float(spatial_scale), float(temporal_scale), float(scale_sigma)
        self.n_bumps, self.noise = n_bumps, float(noise)
        self.min_visible = max(1, int(round(min_visible_frac * self.C)))
        if balance:
            density = np.exp(-(self.D ** 2) / (2 * self.sp ** 2)).sum(1)
            w = 1.0 / density
        else:
            w = np.ones(self.C)
        self.centre_p = w / w.sum()

    def _one(self, rng):
        C, T = self.C, self.T
        a, b = self.ratio * self.conc, (1 - self.ratio) * self.conc
        ratio = float(rng.beta(a, b)) if 0 < self.ratio < 1 else self.ratio
        field = np.zeros((C, T))
        t_axis = np.arange(T)
        for _ in range(int(rng.integers(self.n_bumps[0], self.n_bumps[1] + 1))):
            c = rng.choice(C, p=self.centre_p)
            r = self.sp * np.exp(self.sig * rng.standard_normal())
            L = self.tp * np.exp(self.sig * rng.standard_normal())
            t0 = rng.uniform(-0.5, T - 0.5)
            amp = rng.uniform(0.5, 1.5)
            field += amp * np.exp(-self.D[c] ** 2 / (2 * r ** 2))[:, None] \
                * np.exp(-((t_axis - t0) ** 2) / (2 * L ** 2))[None]
        field = (field - field.mean()) / (field.std() + 1e-9)
        field = field + self.noise * rng.standard_normal((C, T)) + 1e-6 * rng.standard_normal((C, T))
        k = int(round(ratio * C * T))
        k = min(k, (C - self.min_visible) * T)
        mask = np.zeros(C * T, dtype=bool)
        mask[np.argsort(-field.ravel())[:k]] = True
        mask = mask.reshape(C, T)
        # every time token keeps at least `min_visible` electrodes
        for t in range(T):
            over = mask[:, t].sum() - (C - self.min_visible)
            if over > 0:
                hidden = np.flatnonzero(mask[:, t])
                mask[hidden[np.argsort(field[hidden, t])[:over]], t] = False
        return mask

    def __call__(self, batch_size, rng=None):
        """(B, C, T) boolean mask, True = hidden. Pass a numpy Generator for repeatability."""
        rng = rng if rng is not None else np.random.default_rng()
        return np.stack([self._one(rng) for _ in range(batch_size)])


def expand_mask(mask, patch_len, n_samples):
    """(B, C, T) token mask -> (B, C, n_samples) sample mask, each token covering patch_len."""
    out = np.repeat(mask, patch_len, axis=-1)[..., :n_samples]
    if out.shape[-1] < n_samples:                       # pad with the last token's value
        out = np.concatenate([out, np.repeat(out[..., -1:], n_samples - out.shape[-1], -1)], -1)
    return out


import torch.nn as nn
from einops import rearrange
from braindecode.models.reve import FourierEmb4D


def add_lora(net, rank, ffn_rank, ffn_alpha, dropout):
    """LoRA on the attention projections and the two feed-forward matrices of every block (same places as stage 2)."""
    for name, m in list(net.named_modules()):
        if isinstance(m, nn.Linear) and name.split('.')[-1] in ('to_qkv', 'to_out'):
            parent = net.get_submodule(name.rsplit('.', 1)[0])
            setattr(parent, name.split('.')[-1], LoRALinear(m, rank, 2 * rank, dropout))
    if ffn_rank:
        for name, m in list(net.named_modules()):
            if isinstance(m, nn.Linear) and name.startswith('transformer.layers.') and name.endswith(('.net.1', '.net.3')):
                parent = net.get_submodule(name.rsplit('.', 1)[0])
                setattr(parent, name.split('.')[-1], LoRALinear(m, ffn_rank, ffn_alpha, dropout))
    for p_ in net.parameters():
        p_.requires_grad_(False)
    for n_, p_ in net.named_parameters():
        if n_.endswith(('.A', '.B')):
            p_.requires_grad_(True)


def merge_lora(net):
    """Fold every adapter into its base weight (W + scale * B A) and put the plain Linear back. Exact."""
    for name, m in list(net.named_modules()):
        if isinstance(m, LoRALinear):
            m.base.weight.data += (m.B @ m.A * m.scale).to(m.base.weight.dtype)
            parent = net.get_submodule(name.rsplit('.', 1)[0])
            setattr(parent, name.split('.')[-1], m.base)


class MaeDecoder(nn.Module):
    """Takes the encoder's outputs for the visible tokens, fills hidden positions with one learned vector, adds
    position embeddings, runs a small transformer and predicts the raw samples of each hidden patch."""

    def __init__(self, enc_dim, patch, dim=256, depth=4, heads=8):
        super().__init__()
        self.inp, self.pos = nn.Linear(enc_dim, dim), nn.Linear(enc_dim, dim)
        self.mask_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        layer = nn.TransformerEncoderLayer(dim, heads, dim * 4, dropout=0.0, batch_first=True, norm_first=True, activation='gelu')
        self.blocks, self.norm, self.out = nn.TransformerEncoder(layer, depth), nn.LayerNorm(dim), nn.Linear(dim, patch)

    def forward(self, vis, pos_all, b_m, b_u):
        B, N, _ = pos_all.shape
        full = self.mask_token.expand(B, N, -1).contiguous()
        full = full.scatter(1, b_u[..., None].expand(-1, -1, full.shape[-1]), self.inp(vis).to(full.dtype))
        x = self.norm(self.blocks(full + self.pos(pos_all)))
        return self.out(x.gather(1, b_m[..., None].expand(-1, -1, x.shape[-1])))


class MaeLoRA(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=3, n_chans=3, n_times=200, sfreq=200.0,
                                        chs_info=[{'ch_name': c} for c in ('C3', 'Cz', 'C4')])
        self.net.final_layer = nn.Identity()
        add_lora(self.net, S1_RANK, S1_RANK_FFN, S1_ALPHA_FFN, S1_LORA_DROPOUT)
        self.dec = MaeDecoder(self.net.embed_dim, self.net.patch_size, S1_DEC_DIM, S1_DEC_DEPTH)

    def forward(self, eeg, pos, b_m, b_u):
        """eeg (B, C, T) at 200 Hz, pos (B, C, 3), b_m / b_u (B, n) indices into the flattened (channel, patch) tokens.
        Only visible tokens enter the encoder, as in REVE's own pretraining. Returns the L1 loss on hidden patches."""
        n = self.net
        patches = eeg.unfold(2, n.patch_size, n.patch_size - n.patch_overlap)       # (B, C, P, patch)
        B, C, P, _ = patches.shape
        pos4 = FourierEmb4D.add_time_patch(pos, P)
        with torch.no_grad():
            pos_embed = n.ln(n.fourier4d(pos4) + n.mlp4d(pos4))                      # frozen
        tok = rearrange(n.to_patch_embedding(patches), 'b c p e -> b (c p) e') + pos_embed
        vis = n.transformer(tok.gather(1, b_u[..., None].expand(-1, -1, tok.shape[-1])))
        target = patches.reshape(B, C * P, -1).gather(1, b_m[..., None].expand(-1, -1, patches.shape[-1]))
        return torch.nn.functional.l1_loss(self.dec(vis, pos_embed, b_m, b_u).float(), target.float())


def mask_indices(mask, k, g):
    """(B, C, P) bool, True = hidden -> equal-length index arrays (every sample hides exactly k tokens)."""
    flat = mask.reshape(len(mask), -1)
    bm, bu = [], []
    for row in flat:
        hid, vis = np.flatnonzero(row), np.flatnonzero(~row)
        if len(hid) > k:
            mv = g.choice(hid, len(hid) - k, replace=False); hid = np.setdiff1d(hid, mv); vis = np.concatenate([vis, mv])
        elif len(hid) < k:
            mv = g.choice(vis, k - len(hid), replace=False); vis = np.setdiff1d(vis, mv); hid = np.concatenate([hid, mv])
        bm.append(hid); bu.append(vis)
    return torch.as_tensor(np.stack(bm)), torch.as_tensor(np.stack(bu))

In [ ]:
# Stage 1 training, then save.
import json, math, matplotlib.pyplot as plt

if RUN_STAGE1:
    STAGE1_DIR.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(SEED)
    mae = MaeLoRA().to(DEVICE)
    adapters = [p_ for n_, p_ in mae.net.named_parameters() if p_.requires_grad]
    opt = torch.optim.AdamW([dict(params=adapters, lr=0.0, base_lr=S1_LR_ADAPTER),
                             dict(params=list(mae.dec.parameters()), lr=S1_LR_DEC, base_lr=S1_LR_DEC)], weight_decay=S1_WD)
    n_ad = sum(p_.numel() for p_ in adapters)
    print(f'stage 1: {n_ad:,} adapter params + {sum(p_.numel() for p_ in mae.dec.parameters()):,} decoder params, '
          f'{STAGE1_STEPS} steps x batch {STAGE1_BATCH}, mask {S1_MASK_RATIO}, decoder-only for the first {S1_WARMUP} steps')
    rng_np = np.random.default_rng(SEED)
    maskers = {k: ElectrodeMasker(b['pos'], n_time=2, ratio=S1_MASK_RATIO, ratio_concentration=1e6,
                                  spatial_scale=AUG_MASK_RADIUS, noise=AUG_MASK_NOISE) for k, b in S1.items()}
    sizes = np.array([(~b['val']).sum() for b in S1.values()], float); probs = sizes / sizes.sum()
    names = list(S1)

    def batch_of(name, idx, g):
        b = S1[name]
        x = torch.as_tensor(b['X'][idx]).to(DEVICE)
        pos = torch.as_tensor(b['pos'], dtype=torch.float32, device=DEVICE).expand(len(idx), -1, -1)
        C = x.shape[1]; k = int(round(S1_MASK_RATIO * C * 2))
        b_m, b_u = mask_indices(maskers[name](len(idx), g), k, g)
        return x, pos, b_m.to(DEVICE), b_u.to(DEVICE)

    @torch.no_grad()
    def val_loss():
        mae.eval(); tot, cnt = 0.0, 0
        for name in names:
            vi = np.flatnonzero(S1[name]['val'])[:STAGE1_VAL_WINDOWS]
            g = np.random.default_rng(1234)                      # same masks every time
            for i in range(0, len(vi), 64):
                x, pos, b_m, b_u = batch_of(name, vi[i:i + 64], g)
                with torch.autocast('cuda', dtype=torch.bfloat16, enabled=DEVICE == 'cuda'):
                    l_ = mae(x, pos, b_m, b_u)
                tot += float(l_) * len(x); cnt += len(x)
        mae.train(); return tot / cnt

    hist, tr_acc, best, best_state = [], [], 1e9, None
    mae.train()
    for step in range(1, STAGE1_STEPS + 1):
        scale = 0.5 * (1 + math.cos(math.pi * step / STAGE1_STEPS))
        for gi, grp in enumerate(opt.param_groups):
            grp['lr'] = 0.0 if (gi == 0 and step <= S1_WARMUP) else grp['base_lr'] * scale
        name = names[rng_np.choice(len(names), p=probs)]
        pool = np.flatnonzero(~S1[name]['val'])
        x, pos, b_m, b_u = batch_of(name, rng_np.choice(pool, STAGE1_BATCH, replace=False), rng_np)
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=DEVICE == 'cuda'):
            loss = mae(x, pos, b_m, b_u)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(mae.parameters()), 5.0)
        opt.step(); tr_acc.append(float(loss))
        if step % STAGE1_EVAL_EVERY == 0 or step == STAGE1_STEPS:
            vl = val_loss(); hist.append((step, float(np.mean(tr_acc[-STAGE1_EVAL_EVERY:])), vl)); tr_acc_ = hist[-1][1]
            if step > S1_WARMUP and vl < best:
                best = vl; best_state = {k: v.detach().cpu().clone() for k, v in mae.state_dict().items()
                                         if k.startswith('dec.') or k.endswith(('.A', '.B'))}
            print(f'  step {step:5d}  train L1 {tr_acc_:.4f}  held-out L1 {vl:.4f}{"  *" if best_state is not None and vl == best else ""}'
                  f'{"   (adapters still frozen)" if step <= S1_WARMUP else ""}')
    assert best_state is not None, 'no evaluation after the decoder-only warm-up: raise STAGE1_STEPS or lower S1_WARMUP'
    st = np.array(hist)
    plt.figure(figsize=(5.5, 3.6)); plt.plot(st[:, 0], st[:, 1], label='train L1 (masked patches)'); plt.plot(st[:, 0], st[:, 2], label='held-out people L1')
    plt.axvline(S1_WARMUP, ls=':', c='grey'); plt.legend(); plt.xlabel('step'); plt.ylabel('L1'); plt.grid(alpha=0.3); plt.title('stage 1 reconstruction'); plt.show()

    # ---- save: adapters + decoder (small), then the merged encoder (what stage 2 loads)
    mae.load_state_dict(best_state, strict=False)
    torch.save(best_state, STAGE1_DIR / f'{STAGE1_NAME}_adapters_decoder.pt')
    mae.eval()
    probe = torch.as_tensor(next(iter(S1.values()))['X'][:4]).to(DEVICE)
    posb = torch.as_tensor(next(iter(S1.values()))['pos'], dtype=torch.float32, device=DEVICE).expand(4, -1, -1)
    with torch.no_grad():
        before = mae.net(probe, posb, return_features=True)['features']
    merge_lora(mae.net)
    with torch.no_grad():
        after = mae.net(probe, posb, return_features=True)['features']
    err = float((before - after).abs().max())
    print(f'merge check: largest feature difference before vs after folding the adapters in = {err:.2e}')
    assert err < 1e-3, 'merged encoder does not reproduce the adapted one'
    torch.save({k: v.cpu() for k, v in mae.net.state_dict().items() if not k.startswith('final_layer')},
               STAGE1_DIR / f'{STAGE1_NAME}_encoder_merged.pt')
    (STAGE1_DIR / f'{STAGE1_NAME}_card.json').write_text(json.dumps(dict(
        datasets={k: dict(windows=int(len(b['X'])), people=int(len(set(b['subj']))), channels=len(b['chs']), doi=b['doi']) for k, b in S1.items()},
        steps=STAGE1_STEPS, batch=STAGE1_BATCH, mask_ratio=S1_MASK_RATIO, rank=S1_RANK, ffn_rank=S1_RANK_FFN,
        lr_adapter=S1_LR_ADAPTER, lr_decoder=S1_LR_DEC, warmup_steps=S1_WARMUP, best_heldout_l1=best, seed=SEED,
        note='External corpora used for stage 1; declare them (and their licences) in the method description.'), indent=2))
    print('saved to', STAGE1_DIR)
    for f_ in sorted(STAGE1_DIR.glob(f'{STAGE1_NAME}_*')):
        print(f'  {f_.name}  {f_.stat().st_size / 1e6:.0f} MB')
    del mae, opt
else:
    print('Stage 1 training skipped (RUN_STAGE1 = False)')

## Stage 2 — PROTEUS: sealed-like folds with the stage-1 encoder

In [ ]:
import json, os, time, warnings, zipfile
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch
from sklearn.metrics import balanced_accuracy_score

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = Path(ROOT) / 'Track2'
DATA = BASE / 'dreyer2026proteus'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def load_npz(path, tries=5):
    """np.load that survives the Drive mount dropping mid-read (Errno 107): remount,
    then retry."""
    for attempt in range(tries):
        try:
            with np.load(path) as z:
                return {k: z[k] for k in z.files}
        except (OSError, zipfile.BadZipFile, EOFError) as e:
            if attempt == tries - 1 or not ROOT.startswith('/content/drive'):
                raise
            print(f'Drive read failed on {Path(path).name} ({e}); remounting and retrying')
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)


man = json.loads((DATA / 'manifest.json').read_text())
recs = man['recordings']
group = {r['subject']: r['group'] for r in recs.values()}
parts = {g: ([], [], [], [], []) for g in ('train', 'val', 'test')}   # X, y, person, session, run
CH = None
for name, r in sorted(recs.items()):
    z = load_npz(DATA / 'windows' / r['file'])
    p = parts[group[r['subject']]]
    p[0].append(z['X'].astype('float32')); p[1].append(z['y'])
    p[2].append(np.full(len(z['y']), r['subject']))
    p[3].append(np.full(len(z['y']), name))
    n_w = len(z['y'])      # windows are stored in run order; fall back to six equal chunks if no run ids
    p[4].append(np.asarray(z['run']) if 'run' in z else np.repeat(np.arange(1, 7), -(-n_w // 6))[:n_w])
    CH = CH or r['channels']
X, Y, P, S, RUNS = {}, {}, {}, {}, {}
for g, (xs, ys, ps, ss, rs) in parts.items():
    X[g], Y[g], P[g], S[g], RUNS[g] = (np.concatenate(xs), np.concatenate(ys), np.concatenate(ps),
                                       np.concatenate(ss), np.concatenate(rs))
N_CHANS, N_TIMES = X['train'].shape[1:]
N_CLASSES = 3
SFREQ = man['preprocessing']['sfreq']
XF = dict(X)      # full-length raw windows (4 s), kept for the covariance, PSD and streaming checks


def session_norm(Xg, Sg, prior_var, n0=5.0):
    """Causal per-session scaling. Windows of one session are contiguous and in run order. Each window
    is divided, per electrode, by the root of the mean power of the windows BEFORE it in its session
    (blended with a prior of n0 windows), after removing its own mean. Labels are never used."""
    out = np.empty(Xg.shape, dtype='float32')
    _, first = np.unique(Sg, return_index=True)
    for s_ in Sg[np.sort(first)]:
        idx = np.flatnonzero(Sg == s_)
        x = Xg[idx].astype('float64')
        x = x - x.mean(-1, keepdims=True)
        v = x.var(-1)                                   # (windows, channels)
        before = np.cumsum(v, 0) - v                    # power summed over earlier windows only
        t = np.arange(len(idx))[:, None]
        m = (n0 * prior_var[None] + before) / (n0 + t)
        out[idx] = (x / np.sqrt(m)[..., None]).astype('float32')
    return out


if CROP_S:
    # Windows on Drive are the harness's 4 s (1-5 s after the cue). The solver crops the
    # same way, because it already truncates inputs longer than the n_times it trained on.
    keep = int(round(CROP_S * SFREQ))
    assert keep <= N_TIMES, f'CROP_S={CROP_S} longer than the stored {N_TIMES} samples'
    X = {g: a[..., :keep] for g, a in X.items()}
    N_TIMES = keep
prior_var = np.concatenate([a.astype('float64').var(-1) for a in X.values()]).mean(0)   # per electrode
X = {g: session_norm(a, S[g], prior_var, SESSION_N0) for g, a in X.items()}   # what REVE reads
print('windows scaled by the running power of their own session (causal, no labels)')
assert not set(P['test']) & set(P['train']) and not set(P['val']) & set(P['train'])
for g in ('train', 'val', 'test'):
    print(f'{g:5}  {len(set(P[g])):2d} people  {len(Y[g]):6,d} windows   '
          f'mi {(Y[g] == 0).sum():,} / sub {(Y[g] == 1).sum():,} / word {(Y[g] == 2).sum():,}')
print(f'{N_CHANS} channels, {N_TIMES} samples at {SFREQ:g} Hz, device {DEVICE}')


def scores(y, pred):
    return dict(bal_acc=float(balanced_accuracy_score(y, pred)), acc=float((y == pred).mean()))


def null(y, pred, n=200):
    """Balanced accuracy with the predictions shuffled across windows: same predictions,
    no link to the labels. A model has learned something only if it beats this."""
    return np.array([balanced_accuracy_score(y, rng.permutation(pred)) for _ in range(n)])

In [ ]:
# Source of truth for the mask generator is src/eeg_masking.py in the repo.
import numpy as np


class ElectrodeMasker:
    """positions: (C, 3) electrode coordinates in metres (REVE's position bank is in metres,
    head radius about 0.095). n_time: number of time tokens per window."""

    def __init__(self, positions, n_time=1, ratio=0.2, ratio_concentration=8.0,
                 spatial_scale=0.05, temporal_scale=1.0, scale_sigma=0.5,
                 n_bumps=(2, 5), noise=0.3, min_visible_frac=0.4, balance=True):
        pos = np.asarray(positions, dtype=float)
        self.C, self.T = len(pos), int(n_time)
        self.D = np.linalg.norm(pos[:, None] - pos[None], axis=-1)       # (C, C) metres
        self.ratio, self.conc = float(ratio), float(ratio_concentration)
        self.sp, self.tp, self.sig = float(spatial_scale), float(temporal_scale), float(scale_sigma)
        self.n_bumps, self.noise = n_bumps, float(noise)
        self.min_visible = max(1, int(round(min_visible_frac * self.C)))
        if balance:
            density = np.exp(-(self.D ** 2) / (2 * self.sp ** 2)).sum(1)
            w = 1.0 / density
        else:
            w = np.ones(self.C)
        self.centre_p = w / w.sum()

    def _one(self, rng):
        C, T = self.C, self.T
        a, b = self.ratio * self.conc, (1 - self.ratio) * self.conc
        ratio = float(rng.beta(a, b)) if 0 < self.ratio < 1 else self.ratio
        field = np.zeros((C, T))
        t_axis = np.arange(T)
        for _ in range(int(rng.integers(self.n_bumps[0], self.n_bumps[1] + 1))):
            c = rng.choice(C, p=self.centre_p)
            r = self.sp * np.exp(self.sig * rng.standard_normal())
            L = self.tp * np.exp(self.sig * rng.standard_normal())
            t0 = rng.uniform(-0.5, T - 0.5)
            amp = rng.uniform(0.5, 1.5)
            field += amp * np.exp(-self.D[c] ** 2 / (2 * r ** 2))[:, None] \
                * np.exp(-((t_axis - t0) ** 2) / (2 * L ** 2))[None]
        field = (field - field.mean()) / (field.std() + 1e-9)
        field = field + self.noise * rng.standard_normal((C, T)) + 1e-6 * rng.standard_normal((C, T))
        k = int(round(ratio * C * T))
        k = min(k, (C - self.min_visible) * T)
        mask = np.zeros(C * T, dtype=bool)
        mask[np.argsort(-field.ravel())[:k]] = True
        mask = mask.reshape(C, T)
        # every time token keeps at least `min_visible` electrodes
        for t in range(T):
            over = mask[:, t].sum() - (C - self.min_visible)
            if over > 0:
                hidden = np.flatnonzero(mask[:, t])
                mask[hidden[np.argsort(field[hidden, t])[:over]], t] = False
        return mask

    def __call__(self, batch_size, rng=None):
        """(B, C, T) boolean mask, True = hidden. Pass a numpy Generator for repeatability."""
        rng = rng if rng is not None else np.random.default_rng()
        return np.stack([self._one(rng) for _ in range(batch_size)])


def expand_mask(mask, patch_len, n_samples):
    """(B, C, T) token mask -> (B, C, n_samples) sample mask, each token covering patch_len."""
    out = np.repeat(mask, patch_len, axis=-1)[..., :n_samples]
    if out.shape[-1] < n_samples:                       # pad with the last token's value
        out = np.concatenate([out, np.repeat(out[..., -1:], n_samples - out.shape[-1], -1)], -1)
    return out


AUG_ON = AUG_MASK_RATIO > 0 or AUG_GAIN_SD > 0 or AUG_SHIFT_S > 0
NEEDS_TRAINING = RUN_REVE_FOLDS or ZIP_REVE    # the covariance and PSD models do not need REVE
if AUG_MASK_RATIO > 0 and NEEDS_TRAINING:
    _tmp = load_reve(N_CLASSES)                    # only to read REVE's electrode positions
    _folded = {k.casefold(): k for k in _tmp._position_bank.mapping}
    POS = _tmp.get_positions([_folded[c.casefold()] for c in CH]).detach().cpu().numpy().astype(float)
    del _tmp
    TOK = int(SFREQ)                               # one time token = 1 s
    MASKER = ElectrodeMasker(POS, n_time=max(1, -(-N_TIMES // TOK)), ratio=AUG_MASK_RATIO,
                             spatial_scale=AUG_MASK_RADIUS, noise=AUG_MASK_NOISE)


def aug(x):
    """Applied to a standardised training batch (B, C, T). Hidden electrodes become zeros,
    which is also how the solver treats an electrode that is absent at inference."""
    if not AUG_ON:
        return x
    B, C, T = x.shape
    g = np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,))))   # follows torch's seed
    if AUG_GAIN_SD > 0:
        x = x * torch.exp(AUG_GAIN_SD * torch.randn(B, C, 1, device=x.device))
    if AUG_SHIFT_S > 0:
        s = int(round(AUG_SHIFT_S * SFREQ))
        sh = g.integers(-s, s + 1, B).tolist()
        x = torch.stack([torch.roll(x[i], sh[i], dims=-1) for i in range(B)])
    if AUG_MASK_RATIO > 0:
        m = expand_mask(MASKER(B, g), TOK, T)
        x = x.masked_fill(torch.as_tensor(m, device=x.device), 0.0)
    return x


print(f'augmentation: mask {AUG_MASK_RATIO} (noise {AUG_MASK_NOISE}, radius {AUG_MASK_RADIUS} m), '
      f'gain sd {AUG_GAIN_SD}, shift {AUG_SHIFT_S} s' if AUG_ON else 'augmentation off')

In [ ]:
# Shared by the stage-2 folds and the zip: the sealed-like folds, so every method is scored on exactly the same data.
RESULTS = {}      # method name -> (unseen, seen-later, mix)
HIST = {}         # method name -> list of per-epoch histories (one per fold and seed): train loss, held-out loss, accuracy
POOLED = {}       # method name -> held-out balanced accuracy pooled over all held-out windows, per fold
FIT_TRAIN = {}    # method name -> balanced accuracy on its own training windows, per fold (for models without epochs)
import matplotlib.pyplot as plt


def sealed_setup():
    people = sorted(set(P['train']) | set(P['val']) | set(P['test']))
    cat = lambda d: np.concatenate([d[g] for g in ('train', 'val', 'test')])
    sd = dict(Xa=cat(X), XFa=cat(XF), Ya=cat(Y), Pa=cat(P), Sa=cat(S), Ra=cat(RUNS))
    ses_no = np.array([int(str(s_).split('ses-')[1]) for s_ in sd['Sa']])
    later = np.zeros(len(sd['Ya']), bool)         # the data a represented person is tested on
    for p_ in people:
        m_ = sd['Pa'] == p_
        sess_ = sorted(set(ses_no[m_]))
        if len(sess_) > 1:
            later |= m_ & (ses_no > sess_[0])
        else:
            later |= m_ & np.isin(sd['Ra'], sorted(set(sd['Ra'][m_]))[-LATE_RUNS:])
    order = np.random.default_rng(SEED).permutation(people)
    sd.update(later=later, folds=[order[i::5] for i in range(5)])
    return sd


SD = sealed_setup()
_, _first = np.unique(SD['Sa'], return_index=True)
SESSIONS = [np.flatnonzero(SD['Sa'] == s_) for s_ in SD['Sa'][np.sort(_first)]]   # windows of each session, in order


def _cell_scores(rows, pred_of):
    out = []
    for p_, s_ in sorted({(SD['Pa'][r], SD['Sa'][r]) for r in rows}):
        idx = [r for r in rows if SD['Pa'][r] == p_ and SD['Sa'][r] == s_]
        out.append(float(balanced_accuracy_score(SD['Ya'][idx], pred_of[idx])))
    return out


def sealed_eval(name, fit_predict, verbose=True):
    # fit_predict(train_rows, eval_rows) -> class probabilities (len(eval_rows), 3) for the eval rows.
    Pa, later = SD['Pa'], SD['later']
    unseen_all, seen_all, mix_all = [], [], []
    SD['name'] = name
    for f, held in enumerate(SD['folds']):
        unseen = np.isin(Pa, held)
        seen_late = ~unseen & later
        train_m = ~unseen & ~later
        ev = unseen | seen_late
        probs = fit_predict(np.flatnonzero(train_m), np.flatnonzero(ev))
        pred_of = np.full(len(Pa), -1)
        pred_of[np.flatnonzero(ev)] = np.asarray(probs).argmax(1)
        POOLED.setdefault(name, []).append(float(balanced_accuracy_score(SD['Ya'][ev], pred_of[ev])))
        u = _cell_scores(list(np.flatnonzero(unseen)), pred_of)
        sn = _cell_scores(list(np.flatnonzero(seen_late)), pred_of)
        mix = 0.5 * (np.mean(u) + np.mean(sn))
        unseen_all.append(np.mean(u)); seen_all.append(np.mean(sn)); mix_all.append(mix)
        if verbose:
            print(f'  [{name}] fold {f + 1}/5  unseen {np.mean(u):.4f} ({len(u)} cells)   '
                  f'seen-later {np.mean(sn):.4f} ({len(sn)} cells)   mix {mix:.4f}   '
                  f'(trained on {int(train_m.sum())} windows)')
    RESULTS[name] = (float(np.mean(unseen_all)), float(np.mean(seen_all)), float(np.mean(mix_all)))
    return RESULTS[name]


def print_results():
    print('\nsealed-like folds, mean over 5 folds (chance 0.333)')
    print(f'  {"method":58s} {"unseen":>7s} {"seen":>7s} {"MIX":>7s}')
    for k, (u, s_, m_) in RESULTS.items():
        print(f'  {k:58s} {u:7.4f} {s_:7.4f} {m_:7.4f}')


def plot_curves(names, fname='curves_sealed_like.png'):
    # Mean over folds (and seeds) of the per-epoch training loss, held-out loss and held-out accuracy, with the
    # min-max band across folds. Held-out = the unseen people and the later data of the seen people, pooled.
    # Axis limits are fixed (PLOT_LOSS_YLIM, PLOT_ACC_YLIM) so figures from different runs compare.
    names = [n for n in names if HIST.get(n)]
    if not names:
        print('no learning curves were recorded'); return
    fig, axes = plt.subplots(2, len(names), figsize=(4.8 * len(names), 7.2), squeeze=False)
    for j, nm in enumerate(names):
        runs = HIST[nm]
        n_ep = min(len(r) for r in runs)
        ep = np.arange(1, n_ep + 1)
        arr = lambda key: np.array([[h_[key] for h_ in r[:n_ep]] for r in runs])
        tl, vl, ba = arr('train_loss'), arr('val_loss'), arr('bal_acc')
        ax = axes[0, j]
        for a_, lab in ((tl, 'train loss (augmented, smoothed)'), (vl, 'held-out loss')):
            ax.plot(ep, a_.mean(0), label=lab)
            ax.fill_between(ep, a_.min(0), a_.max(0), alpha=0.15)
        ax.axhline(float(np.log(3)), ls='--', c='grey', lw=1, label='chance loss')
        ax.set_ylim(*PLOT_LOSS_YLIM); ax.grid(alpha=0.3); ax.set_xlabel('epoch')
        ax.set_title(f'{nm[:42]}\nfinal gap {vl.mean(0)[-1] - tl.mean(0)[-1]:+.2f}', fontsize=8)
        if j == 0:
            ax.set_ylabel('loss'); ax.legend(fontsize=7, loc='upper right')
        ax = axes[1, j]
        ax.plot(ep, ba.mean(0), c='C2', label='held-out balanced accuracy')
        ax.fill_between(ep, ba.min(0), ba.max(0), color='C2', alpha=0.15)
        ax.axhline(1 / 3, ls='--', c='grey', lw=1, label='chance')
        ax.set_ylim(*PLOT_ACC_YLIM); ax.grid(alpha=0.3); ax.set_xlabel('epoch')
        if j == 0:
            ax.set_ylabel('balanced accuracy'); ax.legend(fontsize=7, loc='upper left')
    fig.tight_layout()
    fig.savefig(BASE / 'weights' / fname, dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
if RUN_REVE_FOLDS:
    Xa_, Ya_ = SD['Xa'], SD['Ya']

    def reve_fit_predict(tr, ev):
        probs_seeds = []
        for s_ in range(SEALED_CV_SEEDS):
            net_f, hist_f, probs_f = train_reve_lora(Xa_[tr], Ya_[tr], Xa_[ev], Ya_[ev], EPOCHS,
                                              'sealed-like fold', quiet=True, seed=SEED + s_)
            HIST.setdefault(SD['name'], []).append(hist_f)
            probs_seeds.append(probs_f(Xa_[ev]))
            del net_f
        return np.mean(probs_seeds, 0)

    keep_before, KEEP = KEEP, 'last'
    REVE_NAME = 'REVE LoRA, causal session scaling' + (f', stage 1 = {STAGE1_NAME}' if USE_STAGE1 else ', plain REVE-base')
    sealed_eval(REVE_NAME, reve_fit_predict)
    KEEP = keep_before
    print_results()
    plot_curves([REVE_NAME], 'curves_reve.png')
else:
    print('Step 2 skipped (RUN_REVE_FOLDS = False)')

## Zip — final run and harness-style checks (REVE with the stage-1 encoder)

Same as `track2_03` Step 4b; the shipped `encoder.pt` is the stage-1 merged encoder when `USE_STAGE1 = True`, so the zip needs no extra files.

In [ ]:
# Source of truth for the REVE solver: solvers/track2_bci_reve_lora.py in the repo.
SUBMISSION_PY_REVE = '"""Track 02 submission: pretrained REVE with LoRA adapters and a head, averaged over seeds.\n\nThe encoder is `brain-bzh/reve-base`. Its weights ship once, in `encoder.pt`, and stay\nfrozen. Each seed ships only its LoRA adapters (rank r, on the 44 attention projections\n`transformer.layers.*.0.{to_qkv,to_out}`, and optionally a weak adapter on the two\nfeed-forward matrices of each block, `transformer.layers.*.1.net.{1,3}`) and its head, in\n`member_<i>.pt`. The head is per-electrode (Track 1\'s), a pooled-query head, or the plain\naverage, as the config says. The\nmembers share one copy of the encoder in memory, because the harness deep-copies the\nsubmitted model for every session.\n\nInput handling repeats the training notebook: electrodes matched to the training ones by\nname (extras ignored, missing ones zero-filled), the window cropped to the first\n`n_times` samples (the notebook\'s CROP_S), per-window per-channel standardisation, then\nresampling from the harness\'s 120 Hz to REVE\'s 200 Hz. With `input_norm: window` the model\nkeeps no state; with `input_norm: session` the windows are scaled by the running power of their own session (causal;\nstate cleared by `reset_state`). Class probabilities are averaged over members before the\nargmax.\n\nThe model definition matches `track2_03_sealed_eval.ipynb`; keep the two in step.\n"""\n\nimport json\nimport os\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import REVE\n\nREVE_SFREQ = 200.0     # REVE\'s pretraining rate\nPATCH = 200            # REVE\'s patch_size in samples (1 s)\nLORA_TARGETS = ("to_qkv", "to_out")\nFFN_TARGETS = (".net.1", ".net.3")       # FeedForward.net is Sequential(norm, Linear, GEGLU, Linear)\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum. Pure torch: nothing is installed at\n    evaluation."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\ndef standardize(x):\n    return (x - x.mean(-1, keepdim=True)) / torch.sqrt(\n        x.var(-1, keepdim=True, unbiased=False) + 1e-8\n    )\n\n\nclass LoRALinear(nn.Module):\n    """Frozen base plus a rank-r update. Names (`base`, `A`, `B`) match training."""\n\n    def __init__(self, base, r, alpha):\n        super().__init__()\n        self.base = base\n        self.A = nn.Parameter(torch.zeros(r, base.in_features))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        self.scale = alpha / r\n\n    def forward(self, x):\n        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale\n\n    @property\n    def weight(self):\n        """Effective weight, W + scale * B A. braindecode >= 2.0 reads `net[1].weight` of the\n        feed-forward GEGLU directly instead of calling the module, which would skip the\n        adapter; with this property the adapter is applied there too. (The input dropout of\n        this adapter does not apply on that path.)"""\n        return self.base.weight + (self.B @ self.A) * self.scale\n\n\nclass PerElectrodeHead(nn.Module):\n    """Shrink each electrode\'s embedding with weights shared across electrodes (one linear\n    layer, or a funnel of several), then a final layer with separate weights per electrode.\n    norm: a LayerNorm after every funnel linear layer, which keeps the signal size steady.\n    init: \'kaiming\' gives the funnel layers He starting weights (PyTorch\'s default shrinks the\n    signal at every layer)."""\n\n    def __init__(self, n_chans, embed_dim, dims, n_outputs, dropout=0.0, mid_dropout=0.0,\n                 norm=False, init=\'default\'):\n        super().__init__()\n        dims = [dims] if isinstance(dims, int) else list(dims)\n        self.norm = nn.LayerNorm(embed_dim)\n        if len(dims) == 1 and not norm:\n            self.shrink = nn.Linear(embed_dim, dims[0])        # keeps the older key names\n        else:\n            layers, prev = [], embed_dim\n            for i, d in enumerate(dims):\n                layers.append(nn.Linear(prev, d))\n                prev = d\n                if norm:\n                    layers.append(nn.LayerNorm(d))\n                if i < len(dims) - 1:\n                    layers += [nn.GELU(), nn.Dropout(mid_dropout)]\n            self.shrink = nn.Sequential(*layers)\n        if init == \'kaiming\':\n            for m in self.shrink.modules():\n                if isinstance(m, nn.Linear):\n                    nn.init.kaiming_normal_(m.weight, nonlinearity=\'relu\')\n                    nn.init.zeros_(m.bias)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(n_chans * dims[-1], n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))\n        return self.out(z.flatten(1))\n\n\nclass MQPoolHead(nn.Module):\n    """K learned queries, each pooling the electrodes by a content-weighted average."""\n\n    def __init__(self, k, embed_dim, n_outputs, dropout=0.0):\n        super().__init__()\n        self.queries = nn.Parameter(torch.randn(k, embed_dim) / embed_dim ** 0.5)\n        self.norm = nn.LayerNorm(embed_dim)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(k * embed_dim, n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.norm(tokens)\n        w = torch.einsum("ke,bce->bkc", self.queries, z) / self.queries.shape[-1] ** 0.5\n        w = w.softmax(dim=-1)\n        pooled = torch.einsum("bkc,bce->bke", w, z)\n        return self.out(self.drop(F.gelu(pooled.flatten(1))))\n\n\ndef is_adapter_key(k):\n    return k.endswith((".A", ".B")) or k.startswith("head.")\n\n\nclass ReveLoRA(nn.Module):\n\n    def __init__(self, channels, n_classes, in_sfreq, rank, encoder_state,\n                 ffn_rank=0, ffn_alpha=0, head_kind="mean", shrink=16, mq_k=2, head_dims=None, head_norm=False):\n        super().__init__()\n        self.head_kind = head_kind\n        self.in_sfreq = float(in_sfreq)\n        self.net = REVE(\n            n_outputs=n_classes, n_chans=len(channels), n_times=PATCH,\n            sfreq=REVE_SFREQ, chs_info=[{"ch_name": c} for c in channels],\n        )\n        self.net.final_layer = nn.Identity()\n        res = self.net.load_state_dict(encoder_state, strict=False)\n        if res.unexpected_keys:\n            raise RuntimeError(f"encoder.pt does not fit REVE: {res.unexpected_keys[:4]}")\n        for p in self.net.parameters():\n            p.requires_grad_(False)\n        names = [\n            n for n, m in self.net.named_modules()\n            if isinstance(m, nn.Linear) and n.split(".")[-1] in LORA_TARGETS\n        ]\n        for name in names:\n            parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n            setattr(parent, name.split(".")[-1],\n                    LoRALinear(self.net.get_submodule(name), rank, 2 * rank))\n        if ffn_rank:                    # the feed-forward matrices, two per block\n            ffn_names = [\n                n for n, m in self.net.named_modules()\n                if isinstance(m, nn.Linear) and n.startswith("transformer.layers.")\n                and n.endswith(FFN_TARGETS)\n            ]\n            for name in ffn_names:\n                parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n                setattr(parent, name.split(".")[-1],\n                        LoRALinear(self.net.get_submodule(name), ffn_rank, ffn_alpha))\n        E = self.net.embed_dim\n        if head_kind == "perelec":\n            self.head = PerElectrodeHead(len(channels), E, head_dims or shrink, n_classes, norm=head_norm)\n        elif head_kind == "mqpool":\n            self.head = MQPoolHead(mq_k, E, n_classes)\n        else:\n            self.head = nn.Linear(E, n_classes)\n\n    def tie_encoder_to(self, other):\n        """Share every frozen tensor with `other`, so ensemble members hold one encoder."""\n        theirs = dict(other.named_parameters())\n        for name, _ in list(self.named_parameters()):\n            if is_adapter_key(name):\n                continue\n            parent = self.get_submodule(name.rsplit(".", 1)[0])\n            setattr(parent, name.rsplit(".", 1)[1], theirs[name])\n\n    def forward(self, x):\n        x = resample_fft(x.float(), int(round(x.shape[-1] * REVE_SFREQ / self.in_sfreq)))\n        feats = self.net(x, return_features=True)["features"]     # (B, chan, patch, embed)\n        if self.head_kind == "mean":\n            return self.head(feats.mean(dim=(1, 2)))\n        return self.head(feats.mean(dim=2))                       # average the time patches only\n\n\nclass NameMatchedEnsemble:\n    """Reorders incoming electrodes into the trained order, zero-filling gaps, and\n    averages the members\' class probabilities."""\n\n    def __init__(self, members, channels, in_names, n_times, device,\n                 input_norm="window", prior_var=None, n0=5.0):\n        self.members, self.device, self.n_times = members, device, int(n_times)\n        self.input_norm, self.n0 = input_norm, float(n0)\n        self.prior_var = None if prior_var is None else torch.as_tensor(prior_var, dtype=torch.float64)\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.src = [lookup.get(c.casefold(), -1) for c in channels]\n        missing = [c for c, i in zip(channels, self.src) if i < 0]\n        print(\n            f"[REVE-LoRA-BCI] matched {len(channels) - len(missing)}/{len(channels)} "\n            f"training channels by name among {len(in_names)} inputs; "\n            f"zero-filled: {missing if missing else \'none\'}; "\n            f"{len(members)} ensemble members"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(\n                f"no training channel found in the input: {list(in_names)[:8]}"\n            )\n        self.reset_state()\n\n    def reset_state(self):\n        # Only the session-scaled model has state: the summed power of the windows seen so far.\n        self.var_sum = torch.zeros(len(self.src), dtype=torch.float64)\n        self.count = 0\n\n    def _session_scale(self, out):\n        """Causal per-session scaling (same as `session_norm` in the training notebook): remove each window\'s\n        mean, divide each electrode by the root of the mean power of the EARLIER windows of the session\n        (blended with a prior of n0 windows), then add this window\'s power to the running sum."""\n        scaled = []\n        for i in range(out.shape[0]):\n            x = out[i].double().cpu()\n            x = x - x.mean(-1, keepdim=True)\n            scale = torch.sqrt((self.n0 * self.prior_var + self.var_sum) / (self.n0 + self.count))\n            scaled.append((x / scale[:, None]).float())\n            self.var_sum = self.var_sum + x.var(-1, unbiased=False)\n            self.count += 1\n        return torch.stack(scaled).to(out.device)\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        if out.shape[-1] > self.n_times:\n            out = out[..., : self.n_times]\n        elif out.shape[-1] < self.n_times:\n            out = F.pad(out, (0, self.n_times - out.shape[-1]))\n        out = self._session_scale(out) if self.input_norm == "session" else standardize(out)\n        probs = torch.stack([m(out).softmax(-1) for m in self.members]).mean(0)\n        return probs.argmax(-1)\n\n\nclass Solver(CompetSolver):\n\n    name = "REVE-LoRA-BCI"\n\n    requirements = ["pip::braindecode"]\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        os.environ["REVE_POSITIONS_PATH"] = str(sub)        # bundled; no download at eval\n        cfg = json.loads((sub / "reve_bci_config.json").read_text())\n        if meta.get("n_classes") not in (None, cfg["n_classes"]):\n            raise RuntimeError(\n                f"trained for {cfg[\'n_classes\']} classes, harness asks for "\n                f"{meta[\'n_classes\']}"\n            )\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        enc_state = torch.load(sub / "encoder.pt", map_location="cpu", weights_only=True)\n        members = []\n        for fname in cfg["member_files"]:\n            m = ReveLoRA(cfg["channels"], cfg["n_classes"], meta["sfreq"],\n                         cfg["rank"], enc_state,\n                         ffn_rank=cfg.get("ffn_rank", 0), ffn_alpha=cfg.get("ffn_alpha", 0),\n                         head_kind=cfg.get("head_kind", "mean"),\n                         shrink=cfg.get("head_shrink", 16), mq_k=cfg.get("mq_k", 2),\n                         head_dims=cfg.get("head_dims"), head_norm=cfg.get("head_norm", False))\n            state = torch.load(sub / fname, map_location="cpu", weights_only=True)\n            res = m.load_state_dict(state, strict=False)\n            absent = [k for k in res.missing_keys if is_adapter_key(k)]\n            if res.unexpected_keys or absent:\n                raise RuntimeError(f"{fname} mismatch: "\n                                   f"{(res.unexpected_keys or absent)[:4]}")\n            if members:\n                m.tie_encoder_to(members[0])\n            members.append(m)\n        members = [m.to(meta["device"]).eval() for m in members]\n        return NameMatchedEnsemble(members, cfg["channels"], in_names,\n                                   cfg["n_times"], meta["device"],\n                                   input_norm=cfg.get("input_norm", "window"),\n                                   prior_var=cfg.get("prior_var"), n0=cfg.get("n0", 5.0))\n'

In [ ]:
if ZIP_REVE:
    import copy, importlib.util, shutil, sys, types
    from braindecode.models import REVE as _REVE
    cat = lambda d: np.concatenate([d[g] for g in ('train', 'val', 'test')])
    Xall, Yall = cat(X), cat(Y)                       # session-scaled windows of all 10 people, all sessions
    keep_before, KEEP = KEEP, 'last'
    finals, prob_fns, curves = [], [], []
    for s_ in range(FINAL_SEEDS):
        # every window is a training window here, so there is no held-out data; the 256 windows passed as
        # "validation" only satisfy the function and mean nothing
        net_s, hist_s, probs_s = train_reve_lora(Xall, Yall, Xall[:256], Yall[:256], EPOCHS,
                                                 f'final fit, seed {s_ + 1}', quiet=True, seed=SEED + s_)
        finals.append(net_s); prob_fns.append(probs_s); curves.append([h['train_loss'] for h in hist_s])
    KEEP = keep_before
    plt.figure(figsize=(5, 3.4))
    for c_ in curves:
        plt.plot(range(1, len(c_) + 1), c_)
    plt.axhline(float(np.log(3)), ls='--', c='grey', lw=1)
    plt.ylim(*PLOT_LOSS_YLIM); plt.grid(alpha=0.3); plt.xlabel('epoch'); plt.ylabel('training loss')
    plt.title('final fit on all 10 people (no held-out data)', fontsize=9); plt.show()

    out = BASE / 'submissions' / ('track2_reve_' + (STAGE1_NAME if USE_STAGE1 else 'session'))
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    os.environ.pop('REVE_POSITIONS_PATH', None)
    enc = load_reve(N_CLASSES)                       # plain REVE-base, or the stage-1 merged encoder
    torch.save({k: v.cpu() for k, v in enc.state_dict().items() if not k.startswith('final_layer')}, out / 'encoder.pt')
    del enc
    files = [f'member_{i}.pt' for i in range(len(finals))]
    for f_, m in zip(files, finals):
        torch.save({k: v.cpu() for k, v in m.state_dict().items() if k.endswith(('.A', '.B')) or k.startswith('head.')}, out / f_)
    (out / 'reve_bci_config.json').write_text(json.dumps(dict(
        channels=list(CH), n_times=int(N_TIMES), sfreq=SFREQ, n_classes=N_CLASSES, rank=LORA_RANK,
        ffn_rank=LORA_RANK_FFN, ffn_alpha=LORA_ALPHA_FFN, head_kind='perelec', head_shrink=HEAD_SHRINK,
        head_dims=list(HEAD_DIMS) if HEAD_DIMS else None, head_norm=HEAD_NORM, mq_k=2,
        input_norm='session', prior_var=[float(v) for v in prior_var], n0=5.0,
        member_files=files, crop_s=CROP_S,
        trained_on=f'PROTEUS, {len(set(SD["Pa"]))} people, all sessions, causal session scaling' + (f'; encoder adapted on MOABB ({STAGE1_NAME})' if USE_STAGE1 else '')), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY_REVE)
    import huggingface_hub
    bank = list((Path.home() / 'mne_data').glob('*reve*positions*.json')) or \
        [Path(huggingface_hub.hf_hub_download('brain-bzh/reve-positions', 'positions.json'))]
    shutil.copy(bank[0], out / 'reve_positions.json')

    # Load it the way the harness does, with a stand-in for the base class.
    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def load(names, n_classes=N_CLASSES):
        return mod.Solver().load_model({'ch_names': list(names), 'chs_info': None, 'sfreq': SFREQ,
                                        'submission_dir': out, 'device': DEVICE, 'n_classes': n_classes})

    def stream(model, idx, names=None, extra=0):
        m_ = copy.deepcopy(model); m_.reset_state()      # a fresh copy per session, as in the harness
        res = []
        for i in idx:
            x = SD['XFa'][i:i + 1]                       # the harness sends the full 4 s window
            if names is not None:
                x = x[:, names]
            if extra:
                x = np.concatenate([x, np.random.default_rng(0).standard_normal((1, extra, x.shape[-1])).astype('float32')], 1)
            res.append(int(m_.predict(torch.tensor(x))[0]))
        return np.array(res)

    model0 = load(CH)
    picks = [SESSIONS[0], SESSIONS[len(SESSIONS) // 2], SESSIONS[-1]]
    n_diff = n_tot = 0
    for idx in picks:
        ref = np.mean([pf(SD['Xa'][idx]) for pf in prob_fns], 0).argmax(1)    # this notebook's answer
        n_diff += int((stream(model0, idx) != ref).sum()); n_tot += len(idx)
    first = picks[0][:30]
    base = stream(model0, first)
    perm = np.random.default_rng(1).permutation(len(CH))
    shuffled = stream(load([CH[i] for i in perm]), first, names=perm)
    wide = stream(load(list(CH) + ['EOG1', 'EMG1']), first, extra=2)
    keep_ch = [c for c in CH if c != 'Cz']
    part = stream(load(keep_ch), first, names=[CH.index(c) for c in keep_ch])
    m_b = copy.deepcopy(model0); m_b.reset_state()
    as_batch = m_b.predict(torch.tensor(SD['XFa'][first])).cpu().numpy()      # one call, windows handled in order
    m_a = copy.deepcopy(model0); m_a.reset_state()
    for i in picks[1][:20]:
        m_a.predict(torch.tensor(SD['XFa'][i:i + 1]))
    m_a.reset_state()
    after_reset = np.array([int(m_a.predict(torch.tensor(SD['XFa'][i:i + 1]))[0]) for i in first])
    t0 = time.time(); stream(model0, first[:15]); ms = (time.time() - t0) / 15 * 1000
    try:
        load(CH, n_classes=4); wrong_cls = False
    except RuntimeError:
        wrong_cls = True
    tol = max(1, int(0.01 * n_tot))
    checks = [
        ('labels in {0,1,2}', set(base.tolist()) <= {0, 1, 2}, f'classes {sorted(set(base.tolist()))}'),
        ('same as this notebook (streamed, one window at a time)', n_diff <= tol,
         f'{n_diff} of {n_tot} windows differ over 3 sessions (allowed {tol})'),
        ('one call equals one window at a time', int((as_batch != base).sum()) <= 1, 'same stream, same answers'),
        ('order-independent', np.array_equal(base, shuffled), 'electrodes shuffled'),
        ('extra electrodes ignored', np.array_equal(base, wide), 'two unknown channels added'),
        ('missing electrode', part.shape == base.shape, 'Cz zero-filled, still runs'),
        ('reset_state clears the session', np.array_equal(base, after_reset), 'a new session starts fresh'),
        ('wrong class count raises', wrong_cls, 'asked for 4 classes: refuses rather than guessing'),
        ('speed', ms < 2000, f'{ms:.0f} ms per window on {DEVICE}'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
        print('Download it from Drive and upload to Codabench (Track 2).')
    else:
        print('\nNot zipping: a check failed.')
else:
    print('REVE zip skipped (ZIP_REVE = False)')